This notebook is going to be similar to the quickstart beginner but more in depth explanation of tensorflow 

Lets get started

In [53]:
import tensorflow as tf

In [54]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D
from tensorflow.keras import Model

In [55]:
#Load and prep the data
mnist = tf.keras.datasets.mnist

In [56]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train, X_test = X_train / 255.0, X_test / 255.0

In [57]:
#Channels dimension
# ... is a spread operator it apply for all the elments

X_train = X_train[..., tf.newaxis].astype("float32")
X_test = X_test[..., tf.newaxis].astype("float32")

In [58]:
#Batching and shuffling the dataset
train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
).shuffle(10000).batch(32)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

In [59]:
#Builfing A model pipeline Yay!

class MyModel(Model):
  def __init__(self):
    super(MyModel, self).__init__()
    self.conv1 = Conv2D(32, 3, activation='relu')
    self.flatten = Flatten()
    self.d1 = Dense(128, activation='relu')
    self.d2 = Dense(10)
  def call(self, x):
    x = self.conv1(x)
    x = self.flatten(x)
    x = self.d1(x)
    x = self.d2(x)
    return x

model = MyModel()

In [46]:
#Optimizer and loss funciton
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

optimizer = tf.keras.optimizers.Adam()

In [60]:
#measuring loss and accuracy
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='test_accuracy')


In [29]:
!pip install --upgrade keras

  Found existing installation: Keras 2.3.1
    Uninstalling Keras-2.3.1:
      Successfully uninstalled Keras-2.3.1


In [61]:
#Traing the model using gradient descent
#tf.GradientTape
@tf.function
def train_step(images, labels):
  with tf.GradientTape() as tape:
    # training=True is only needed if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    predictions = model(images, training=True)
    loss = loss_object(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  train_loss(loss)
  train_accuracy(labels, predictions)



In [66]:
#Test the model
@tf.function
def test_step(images, labels):
  # training=False is only needed if there are layers with different
  # behavior during training versus inference (e.g. Dropout).
  predictions = model(images, training=False)
  t_loss = loss_object(labels, predictions)

  test_loss(t_loss)
  test_accuracy(labels, predictions)


In [67]:
#Training the model multiple times
EPOCHS = 5

for epoch in range(EPOCHS):
  #Reset the traing and testing metrics
  train_loss.reset_states()
  train_accuracy.reset_states()
  test_loss.reset_states()
  test_accuracy.reset_states()
  #Go through all the images and the labels
  #Simuntanelously and train it
  for images, labels in train_ds:
    train_step(images, labels)
  
  for test_images, test_labels in test_ds:
    test_step(test_images, test_labels)
  
  template = 'EPOCH {}, Train Loss: {}, Train Accuracy:{}, Test Loss:{}, Test Accuracy:{}'

  print(template.format(epoch + 1,
                        train_loss.result(),
                        train_accuracy.result() * 100,
                        test_loss.result(),
                        test_accuracy.result() * 100))



EPOCH 1, Train Loss: 0.01492052897810936, Train Accuracy:99.55500030517578, Test Loss:0.05848587676882744, Test Accuracy:98.29999542236328
EPOCH 2, Train Loss: 0.008956203237175941, Train Accuracy:99.71666717529297, Test Loss:0.05912931263446808, Test Accuracy:98.44999694824219
EPOCH 3, Train Loss: 0.00655046571046114, Train Accuracy:99.7733383178711, Test Loss:0.056179389357566833, Test Accuracy:98.5999984741211
EPOCH 4, Train Loss: 0.00497463857755065, Train Accuracy:99.83499908447266, Test Loss:0.06590672582387924, Test Accuracy:98.37999725341797
EPOCH 5, Train Loss: 0.0039342474192380905, Train Accuracy:99.87000274658203, Test Loss:0.07465877383947372, Test Accuracy:98.3699951171875
